# Part 1: Data Preprocessing
**Project:** Employee Attrition Prediction in the Saudi Private Sector

**Course:** ARTI 506 

we prepare the raw 2025 Saudi Employee Attrition dataset for machine learning. We will handle missing values, encode categorical features, and scale numerical data. Because we are comparing different algorithms later in the project, outputs two distinct datasets:
1. `processed_dataset_tree.csv` (Label Encoded, Unscaled for Random Forest/Decision Trees)
2. `processed_dataset_nontree.csv` (One-Hot Encoded, Scaled for Logistic Regression/SVM)

In [19]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load the Data
df = pd.read_excel('../Group3_Dataset_of_Employee_Attrition.xlsx')

# 2. Initial Cleanup
# Drop columns that have no predictive power (like Employee ID) if they exist
if 'EmployeeID' in df.columns:
    df = df.drop('EmployeeID', axis=1)

# Separate Target from Features
# Assuming the target column is named 'Attrition'
target_col = 'Attrition'
y = df[target_col]
X = df.drop(target_col, axis=1)

# Strip whitespace and Encode Target (Yes -> 1, No -> 0)
y = y.str.strip().map({'Yes': 1, 'No': 0})


# 3. Handle Missing Values
# Impute numerical columns with the median (robust to outliers)
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

# Impute categorical columns with the mode (most frequent value)
cat_cols = X.select_dtypes(include=['object', 'category']).columns
X[cat_cols] = X[cat_cols].fillna(X[cat_cols].mode().iloc[0])

# 4. Create Tree-Based Dataset (Label Encoding, No Scaling)
X_tree = X.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    X_tree[col] = le.fit_transform(X_tree[col])
    label_encoders[col] = le

df_tree = X_tree.copy()
df_tree[target_col] = y
# Export processed dataset for tree models
df_tree.to_csv('processed_dataset_tree.csv', index=False)

# 5. Create Non-Tree-Based Dataset (One-Hot Encoding, Standard Scaling)
X_non_tree = X.copy()

# One-Hot Encode categorical variables
X_non_tree = pd.get_dummies(X_non_tree, columns=cat_cols, drop_first=True)

# Scale numerical features
scaler = StandardScaler()
# Scale all columns in the non-tree dataset to ensure uniform impact
X_non_tree_scaled = pd.DataFrame(scaler.fit_transform(X_non_tree), columns=X_non_tree.columns)

df_non_tree = X_non_tree_scaled.copy()
df_non_tree[target_col] = y.reset_index(drop=True) # Ensure indices align
# Export processed dataset for non-tree models
df_non_tree.to_csv('processed_dataset_nontree.csv', index=False)

print("Preprocessing complete! Both datasets have been exported successfully.")

C:\Users\haide\AppData\Local\Temp\ipykernel_29512\85651938.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns


Preprocessing complete! Both datasets have been exported successfully.


### Verification
Preview the first few rows of our newly processed datasets to ensure the transformations and scaling were applied correctly.

In [20]:
print("--- Tree-Based Dataset Preview ---")
display(df_tree.head())

print("\n--- Non-Tree-Based Dataset Preview ---")
display(df_non_tree.head())

--- Tree-Based Dataset Preview ---


,ID,Gender,Age,Maritalstatus,Academic_degree,Years_Experience,Years_experience_lastorganization,Sector,Department,JobTitle,...,Distance_to_work,Work_Live_Balance,Physical_Stress,Psychological_Exhaustion,Job_Stability,Health_Issues,Environment_Satisfaction,Job_Satisfaction,Job_Opportunities,Attrition
0,11,1,1,1,1,0,1,11,5,1,...,1,0,2,0,0,1,2,0,1,1
1,12,1,0,2,1,7,1,11,5,1,...,1,0,0,0,1,0,2,1,0,0
2,13,1,0,2,0,7,1,3,21,70,...,0,0,2,1,0,0,2,1,1,0
3,14,1,0,2,0,7,1,3,21,70,...,1,0,2,2,0,0,1,0,0,0
4,15,1,0,2,0,7,1,3,21,70,...,1,0,1,1,0,0,2,0,0,0



--- Non-Tree-Based Dataset Preview ---


,ID,Allowances,Gender _Female,Age_31 to 40,Age_41 to 50,Age_51 to 60,Maritalstatus_ Married,Maritalstatus_ Single,Maritalstatus_Married,Academic_degree _ Master's,...,Psychological_Exhaustion_ Sometimes,Psychological_Exhaustion_ Yes,Job_Stability_ Yes,Health_Issues_ Yes,Environment_Satisfaction_Low,Environment_Satisfaction_Medium,Job_Satisfaction_ Satisfied,Job_Satisfaction_ Very satisfied,Job_Opportunities_Yes,Attrition
0,-1.736682,0.655020,0.869849,1.262843,-0.345486,-0.105051,0.898490,-0.821648,-0.152302,2.644483,...,-0.751695,-0.716479,-0.962904,2.273980,-0.455972,0.868361,-1.088725,-0.518045,1.017790,1
1,-1.733797,0.655020,0.869849,-0.791864,-0.345486,-0.105051,-1.112979,1.217066,-0.152302,2.644483,...,-0.751695,-0.716479,1.038525,-0.439758,-0.455972,0.868361,0.918506,-0.518045,-0.982521,0
2,-1.730912,-1.029923,0.869849,-0.791864,-0.345486,-0.105051,-1.112979,1.217066,-0.152302,-0.378146,...,1.330326,-0.716479,-0.962904,-0.439758,-0.455972,0.868361,0.918506,-0.518045,1.017790,0
3,-1.728026,-1.029923,0.869849,-0.791864,-0.345486,-0.105051,-1.112979,1.217066,-0.152302,-0.378146,...,-0.751695,1.395715,-0.962904,-0.439758,2.193116,-1.151595,-1.088725,-0.518045,-0.982521,0
4,-1.725141,-1.029923,0.869849,-0.791864,-0.345486,-0.105051,-1.112979,1.217066,-0.152302,-0.378146,...,1.330326,-0.716479,-0.962904,-0.439758,-0.455972,0.868361,-1.088725,-0.518045,-0.982521,0
